# Réemploi des plaques gravées entre éditeurs

Certains jeux de plaques gravées (identifiés par le graveur, ex. "Anonyme1572") sont réutilisés
d'une édition à l'autre : soit repris par le même éditeur des années plus tard (réédition avec
les mêmes gravures), soit transmis à un autre éditeur (achat des plaques, héritage...). Exemple :
"Anonyme1572" est publié par Francesco de' Franceschi (Venise, 1572), puis les mêmes plaques
réapparaissent chez Pietro Deuchino en 1587 et 1588.

Cette frise trace, pour chaque jeu de plaques réemployé, la chronologie de ses éditeurs — sur
l'ensemble du corpus (19 villes), pas seulement Lyon/Paris/Venise comme dans
`02_nuage_editions_villes.ipynb`.

Même source de données que `01_carte_circulation.ipynb` et `02_nuage_editions_villes.ipynb` :
`retours_celine/BNU_corpus.ods` (feuille `Synthèse`).

In [ ]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "reemploi_plaques.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS cellule par cellule que les deux autres notebooks (voir
`01_carte_circulation.ipynb` pour le détail des limites de `pandas.read_excel` sur ce fichier).
Contrairement à `02_nuage_editions_villes.ipynb`, on garde ici **toutes** les villes du corpus :
ce n'est pas une question géographique, c'est une question de filiation entre éditeurs à
travers le temps. Seules les éditions avec un graveur identifiable sont retenues — sans
identité de plaques, il n'y a rien à tracer.

In [ ]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence (même logique que
    01_carte_circulation.ipynb et 02_nuage_editions_villes.ipynb)."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def graveur_ou_inconnu(g):
    """None pour un graveur non identifié ('?', 'inaccessible', case vide) : ces éditions sont
    écartées plus bas, contrairement à 02_nuage_editions_villes.ipynb qui leur donnait un repli
    textuel — ici pas d'identité de plaques, pas de ligne dans la frise."""
    g = (g or "").strip()
    if not g or g.lower() in {"?", "inaccessible"}:
        return None
    return g

def graveur_multiple(g):
    """True si le champ graveur nomme plusieurs personnes (crédit composite), qu'elles soient
    séparées par "/" (ex. "Chauveau, François / Lepautre, Jean / Leclerc, Sébastien") ou par
    " et " (ex. "Clein, Francisco (inv.) et Savery, Salomon (sculp.)", "Mathieu, Jean et
    autres"). Une virgule seule ne suffit pas à le détecter : c'est aussi le séparateur
    nom/prénom d'une personne unique (ex. "Goltzius, Hendrick (1558-1617)." doit rester
    inclus). Ces éditions à plusieurs mains sont écartées : le jeu de plaques n'est plus
    attribuable à un seul graveur, donc pas traçable comme une identité unique dans la frise."""
    return "/" in g or re.search(r"\bet\b", g, re.IGNORECASE) is not None

def categorie_technique(t):
    t = (t or "").strip().lower()
    if t == "bois":
        return "bois"
    if t == "cuivre":
        return "cuivre"
    return "inconnue"  # vide, "inaccessible", "?", etc. -> repli neutre plutôt que fragmenter

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

def fusionner_tomes(corpus):
    """Une même édition est parfois scindée en plusieurs tomes dans le tableau source (une
    ligne par tome : même ville/année/éditeur/titre abrégé/graveur, seul le titre complet
    varie selon le tome — ex. Lyon 1697 "Les Oeuvres d'Ovide..." en 3 tomes, Amsterdam 1693 en
    3 tomes). On les fusionne en une seule édition : sinon elles compteraient 2 ou 3 fois pour
    ce qui est en réalité une seule publication. Le lien et le commentaire de copie, souvent
    renseignés sur un seul des tomes, sont récupérés du premier tome qui les a."""
    groupes, ordre = {}, []
    for ligne in corpus:
        cle = (ligne.get("ville", ""), ligne.get("année", ""), ligne.get("publisher", ""),
               ligne.get("titre abrégé", ""), ligne.get(COL_GRAVEUR, ""))
        if cle not in groupes:
            groupes[cle] = []
            ordre.append(cle)
        groupes[cle].append(ligne)

    champs_premier_non_vide = ["url catalogue", "version numérisée 1", "version numérisée 2",
                                "Biblioteca Digital Ovidiana", "copies de cette édition"]
    fusionne = []
    for cle in ordre:
        lignes_tomes = groupes[cle]
        base = dict(lignes_tomes[0])
        if len(lignes_tomes) > 1:
            for champ in champs_premier_non_vide:
                for ligne in lignes_tomes:
                    if ligne.get(champ, "").strip():
                        base[champ] = ligne[champ]
                        break
        fusionne.append(base)
    return fusionne

nb_avant_fusion = len(corpus)
corpus = fusionner_tomes(corpus)
if len(corpus) != nb_avant_fusion:
    print(nb_avant_fusion - len(corpus), "lignes fusionnées (tomes d'une même édition regroupés)")

editions = []
nb_graveurs_multiples = 0
for row in corpus:
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    graveur = graveur_ou_inconnu(row.get(COL_GRAVEUR, ""))
    if graveur is None:
        continue
    if graveur_multiple(graveur):
        nb_graveurs_multiples += 1
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    editions.append({
        "ville": row.get("ville", "").strip() or "Ville inconnue",
        "annee": annee,
        "titre": titre.strip(),
        "technique": categorie_technique(row.get("technique", "")),
        "graveur": graveur,
        "publisher": row.get("publisher", "").strip() or "Éditeur non identifié",
        "lien": extraire_lien(row),
        "copies": row.get("copies de cette édition", "").strip(),
    })

print(len(editions), "éditions avec un graveur identifiable, sur", len(corpus), "au total")
if nb_graveurs_multiples:
    print(nb_graveurs_multiples, "éditions écartées (graveur composite, plusieurs personnes citées)")
print(len({e['graveur'] for e in editions}), "jeux de plaques distincts")

## 2. Choix de visualisation


In [ ]:
groupes_plaques = {}
for e in editions:
    groupes_plaques.setdefault(e["graveur"], []).append(e)

plaques_reemployees = {
    g: sorted(u, key=lambda e: e["annee"])
    for g, u in groupes_plaques.items() if len(u) > 1
}

def editeur_fiable(pub):
    """Un éditeur non identifié ('s.n.', case vide) ne permet pas de dire avec certitude si
    deux éditions se succèdent chez le même éditeur ou changent de main."""
    return pub not in {"s.n.", "Éditeur non identifié"}

def type_segment(pub1, pub2):
    if not (editeur_fiable(pub1) and editeur_fiable(pub2)):
        return "incertain"
    return "reprise" if pub1 == pub2 else "transfert"

ordre_plaques = sorted(
    plaques_reemployees.items(),
    key=lambda kv: (-len(kv[1]), kv[1][0]["annee"])
)

plaques = []
for rang, (graveur, eds) in enumerate(ordre_plaques):
    segments = [
        {"an1": a["annee"], "an2": b["annee"], "type": type_segment(a["publisher"], b["publisher"])}
        for a, b in zip(eds, eds[1:])
    ]
    plaques.append({"graveur": graveur, "rang": rang, "editions": eds, "segments": segments})

nb_transferts = sum(1 for p in plaques for s in p["segments"] if s["type"] == "transfert")
nb_reprises = sum(1 for p in plaques for s in p["segments"] if s["type"] == "reprise")
nb_incertains = sum(1 for p in plaques for s in p["segments"] if s["type"] == "incertain")

print(len(groupes_plaques), "jeux de plaques au total —", len(plaques), "réemployés (≥2 éditions), affichés ci-dessous")
print(nb_transferts, "transmissions à un autre éditeur,", nb_reprises, "réimpressions par le même éditeur,",
      nb_incertains, "cas incertains (éditeur non identifié d'un côté ou de l'autre)")
print()
for p in plaques[:10]:
    chaine = " → ".join(f"{e['publisher']} ({e['annee']})" for e in p["editions"])
    print(f"  {p['graveur']:30s} {chaine}")

## 3. Génération de la frise (HTML autonome)

Une seule grande frise SVG, une ligne par jeu de plaques réemployé, un bandeau alterné ("zèbre") pour repérer facilement
une ligne d'une autre. Légende des trois types de segment, infobulle au survol/clic identique aux deux autres notebooks, 
et un tableau détaillé dépliable listant toutes les éditions de chaque jeu de plaques dans l'ordre chronologique.

In [ ]:
import math

LIBELLES_TECHNIQUE = {"bois": "Bois", "cuivre": "Cuivre", "inconnue": "Technique inconnue"}

FRISE_ANNEE_MIN, FRISE_ANNEE_MAX = 1490, 1750
FRISE_TICKS = [1500, 1600, 1700]
# Échelle temporelle large : à LARGEUR=1000 deux éditions à un an d'écart (ex. 1587/1588)
# tombaient à ~3px l'une de l'autre, quasi confondues avec leurs flèches. Le SVG est rendu en
# pleine largeur de page (voir .frise, section 3 : width:100% + viewBox, pas de barre de
# défilement), donc ces unités ne sont pas des pixels écran fixes — augmenter LARGEUR revient
# à zoomer l'échelle temporelle, la mise à l'échelle finale dépend de la largeur du navigateur.
LARGEUR = 2600
MARGE = {"gauche": 210, "droite": 20, "haut": 30, "bas": 36}
HAUTEUR_LIGNE = 24
RAYON_POINT = 6
ECART_MIN = 2 * RAYON_POINT + 4  # séparation horizontale minimale entre deux points d'une même ligne

# Copie locale : cette cellule ajoute des lignes "solo" à `plaques` (voir plus bas). Sans
# cette copie, ré-exécuter la cellule plusieurs fois sans repasser par la cellule précédente
# dupliquerait ces lignes à chaque nouvelle exécution (elles s'ajouteraient sur la liste
# globale `plaques` au lieu d'en repartir à zéro).
plaques = list(plaques)

def frise_x(annee):
    t = (annee - FRISE_ANNEE_MIN) / (FRISE_ANNEE_MAX - FRISE_ANNEE_MIN)
    return MARGE["gauche"] + t * (LARGEUR - MARGE["gauche"] - MARGE["droite"])

def y_ligne(rang):
    return MARGE["haut"] + rang * HAUTEUR_LIGNE + HAUTEUR_LIGNE / 2

def positions_x_plaque(eds):
    """Position x par édition d'un même jeu de plaques (eds trié par année). À cette échelle,
    même deux éditions à quelques années d'écart (voire la même année, ex. "Anonyme1693" : 3
    éditions en 1693) peuvent tomber plus près que le diamètre d'un point (2×RAYON_POINT) —
    sans correctif elles se chevauchent et on ne distingue plus ni les points ni les segments
    qui les relient. On pousse donc chaque point vers la droite du minimum nécessaire par
    rapport au précédent (jamais vers la gauche : l'ordre chronologique visuel reste intact)."""
    xs = []
    x_precedent = None
    for e in eds:
        x = frise_x(e["annee"])
        if x_precedent is not None and x - x_precedent < ECART_MIN:
            x = x_precedent + ECART_MIN
        xs.append(x)
        x_precedent = x
    return xs

def retrait_vers(x1, y1, x2, y2, retrait):
    """Recule (x2, y2) de `retrait` unités le long du segment (x1,y1)->(x2,y2). Utilisé pour
    les flèches de copie (section suivante) : contrairement aux segments de réemploi, elles
    peuvent être diagonales (deux jeux de plaques différents = deux lignes différentes), donc
    le simple retrait horizontal des flèches de transfert (x2 - constante) ne suffit pas."""
    dx, dy = x2 - x1, y2 - y1
    dist = math.hypot(dx, dy)
    if dist == 0:
        return x2, y2
    t = max(dist - retrait, 0) / dist
    return x1 + dx * t, y1 + dy * t

# --- Fonctions de résolution des copies (mêmes que 01_carte_circulation.ipynb) : déplacées
# ici, avant la construction des lignes, car on en a besoin dès maintenant pour repérer les
# jeux de plaques à édition unique qui doivent malgré tout obtenir une ligne "solo" (voir
# juste après) — donc avant même de calculer la hauteur totale de la frise.
def normaliser_candidat_anonyme(c):
    c = re.sub(r"anonyme\s*(\d{4})", r"Anonyme\1", c, flags=re.IGNORECASE)
    if re.fullmatch(r"\d{4}", c.strip()):
        c = "Anonyme" + c.strip()
    return c.strip()

def eclater_enumeration(fragment):
    parties = [p.strip() for p in fragment.split(",")]
    if len(parties) > 1 and all(re.fullmatch(r"(anonyme\s*)?\d{4}", p, re.IGNORECASE) for p in parties):
        return parties
    return [fragment]

def mention_ambigue(texte):
    """"ou" ou "?" signale une attribution hésitante entre plusieurs graveurs : on ne devine pas."""
    return bool(re.search(r"\bou\b", texte, re.IGNORECASE)) or "?" in texte

def extraire_candidats_copie(texte):
    t = re.sub(r"\(.*?\)", "", texte)
    t = re.sub(r"^\s*(copie|même famille que)\s*", "", t, flags=re.IGNORECASE)
    bruts = re.split(r"\s+et\s+", t)
    candidats = []
    for c in bruts:
        c = c.strip(" .,;?\xa0")
        if not c:
            continue
        for sous in eclater_enumeration(c):
            sous = normaliser_candidat_anonyme(sous.strip(" .,;?\xa0"))
            if sous:
                candidats.append(sous)
    return candidats

def tokens_nom(nom):
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.replace(",", " ")
    return set(t.lower() for t in re.findall(r"[a-zà-öø-ÿ']+", nom) if len(t) >= 3)

def trouver_graveur_connu(candidat, registre):
    if candidat in registre:
        return candidat
    tc = tokens_nom(candidat)
    if not tc:
        return None
    meilleur, meilleur_score = None, 0
    for nom_reg in registre:
        if nom_reg.lower().startswith("anonyme"):
            continue
        score = len(tc & tokens_nom(nom_reg))
        if score > meilleur_score:
            meilleur, meilleur_score = nom_reg, score
    return meilleur if meilleur_score >= 1 else None

registre_graveurs = sorted({e["graveur"] for e in editions})

# --- Lignes "solo" : un jeu de plaques à édition unique reste normalement hors de la frise
# (le seuil ≥2 éditions ne concerne que le RÉEMPLOI). Mais si cette édition unique est
# impliquée dans une copie résoluble — comme source (elle copie un graveur connu) ou comme
# cible (un autre graveur la copie) — elle a quand même besoin d'un point pour que la flèche
# ait une extrémité où s'accrocher. On détecte ces cas ici, avant de figer les lignes.
graveurs_solo_necessaires = {}
for e in editions:
    texte = e.get("copies", "")
    if not texte or mention_ambigue(texte):
        continue
    for c in extraire_candidats_copie(texte):
        match = trouver_graveur_connu(c, registre_graveurs)
        if match is None:
            continue
        instances_cible = groupes_plaques.get(match, [])
        anterieures = [inst for inst in instances_cible if inst["annee"] <= e["annee"] and inst is not e]
        if not anterieures:
            continue
        # Cette copie est résoluble en principe (source et cible existent, dans le bon ordre
        # chronologique) : source et cible ont chacune besoin d'un point affiché.
        if len(groupes_plaques.get(e["graveur"], [])) == 1:
            graveurs_solo_necessaires[e["graveur"]] = groupes_plaques[e["graveur"]]
        if len(instances_cible) == 1:
            graveurs_solo_necessaires[match] = instances_cible

for graveur, eds_solo in sorted(graveurs_solo_necessaires.items()):
    plaques.append({"graveur": graveur, "rang": len(plaques), "editions": eds_solo, "segments": []})

if graveurs_solo_necessaires:
    print(len(graveurs_solo_necessaires), "jeu(x) de plaques à édition unique ajouté(s) en ligne \"solo\","
          "uniquement pour porter une flèche de copie :", ", ".join(sorted(graveurs_solo_necessaires)))

HAUTEUR_PLOT = len(plaques) * HAUTEUR_LIGNE
HAUTEUR = MARGE["haut"] + HAUTEUR_PLOT + MARGE["bas"]

# --- Construction du SVG : bandes zébrées + étiquettes, grille des années, segments, flèches
# de copie, points --- (dans cet ordre : les segments passent par-dessus les bandes/la
# grille, les flèches de copie par-dessus les segments, et les points par-dessus tout, pour
# que rien ne cache leur centre cliquable)
elements_svg = []

for p in plaques:
    y0 = MARGE["haut"] + p["rang"] * HAUTEUR_LIGNE
    if p["rang"] % 2 == 1:
        elements_svg.append(f'<rect x="0" y="{y0}" width="{LARGEUR}" height="{HAUTEUR_LIGNE}" class="bande-zebra"/>')
    elements_svg.append(
        f'<text x="8" y="{y_ligne(p["rang"]) + 4:.1f}" class="etiquette-plaque">{p["graveur"]}</text>'
    )

for t in FRISE_TICKS:
    x = frise_x(t)
    elements_svg.append(f'<line x1="{x:.1f}" y1="{MARGE["haut"]}" x2="{x:.1f}" y2="{MARGE["haut"] + HAUTEUR_PLOT:.1f}" class="grille-frise"/>')
    elements_svg.append(f'<text x="{x:.1f}" y="{MARGE["haut"] + HAUTEUR_PLOT + 18:.1f}" text-anchor="middle" class="etiquette-annee-frise">{t}</text>')

for p in plaques:
    y = y_ligne(p["rang"])
    xs = positions_x_plaque(p["editions"])
    for k, s in enumerate(p["segments"]):
        x1, x2 = xs[k], xs[k + 1]
        marqueur = ' marker-end="url(#fleche-transfert)"' if s["type"] == "transfert" else ""
        # Les points sont dessinés par-dessus les traits (voir plus bas) : sans ce retrait, la
        # pointe d'une flèche de transfert finirait exactement au centre du point d'arrivée et
        # serait cachée dessous. On arrête le trait un peu avant le bord du point pour que la
        # pointe reste visible, juste à son entrée.
        x2_trace = x2 - (RAYON_POINT + 4) if s["type"] == "transfert" else x2
        elements_svg.append(
            f'<line x1="{x1:.1f}" y1="{y:.1f}" x2="{x2_trace:.1f}" y2="{y:.1f}" class="segment-{s["type"]}"{marqueur}/>'
        )

# `points` : une entrée par édition affichée (données seulement — les cercles SVG sont ajoutés
# plus bas, après les flèches de copie, pour rester visuellement au-dessus). L'ordre de cette
# liste est celui de l'attribut data-i de chaque cercle, utilisé par l'infobulle en JS (même
# mécanisme que 02_nuage_editions_villes.ipynb).
points = []
for p in plaques:
    y = y_ligne(p["rang"])
    xs = positions_x_plaque(p["editions"])
    for j, e in enumerate(p["editions"]):
        points.append({
            "fx": round(xs[j], 1),
            "fy": round(y, 1),
            "graveur": p["graveur"],
            "annee": e["annee"],
            "titre": e["titre"],
            "ville": e["ville"],
            "editeur": e["publisher"],
            "technique": e["technique"],
            "lien": e["lien"],
        })

# --- Flèches de copie (colonne `copies de cette édition`) : même logique d'extraction et de
# résolution que 01_carte_circulation.ipynb (section 3 de ce notebook), mais reliant ici deux
# POINTS DE LA FRISE plutôt que deux villes — souvent sur deux lignes différentes (un jeu de
# plaques copié n'est presque jamais le même que celui qui copie), donc tracées en courbe
# plutôt qu'en trait droit pour rester lisibles en traversant les lignes intermédiaires.
# Grâce aux lignes "solo" ajoutées plus haut, une copie résoluble a maintenant toujours un
# point à chaque extrémité (sinon elle n'aurait pas généré de ligne solo) : il ne reste donc
# ci-dessous que les vrais cas non résolubles (mention ambiguë, aucune correspondance, pas
# d'édition antérieure du graveur cité).
points_par_graveur = {}
for i, pt in enumerate(points):
    points_par_graveur.setdefault(pt["graveur"], []).append((pt["annee"], i))
for g in points_par_graveur:
    points_par_graveur[g].sort()

def point_source(e):
    for annee, i in points_par_graveur.get(e["graveur"], []):
        if annee == e["annee"]:
            return i
    return None

fleches_copies = []
non_resolus_copies = []
for e in editions:
    texte = e.get("copies", "")
    if not texte:
        continue
    i_source = point_source(e)
    if i_source is None:
        non_resolus_copies.append((e, texte, "(source)", "édition source non affichée (jeu de plaques jamais réemployé)"))
        continue
    if mention_ambigue(texte):
        non_resolus_copies.append((e, texte, "(toute la mention)", "attribution ambiguë (ou/possibilité multiple)"))
        continue
    matches_uniques = {}
    for c in extraire_candidats_copie(texte):
        match = trouver_graveur_connu(c, registre_graveurs)
        if match is None:
            non_resolus_copies.append((e, texte, c, "aucune correspondance"))
        else:
            matches_uniques.setdefault(match, []).append(c)
    for match in matches_uniques:
        cibles = points_par_graveur.get(match)
        if not cibles:
            non_resolus_copies.append((e, texte, match, "graveur cible non affiché (jamais réemployé)"))
            continue
        anterieures = [(an, i) for an, i in cibles if an <= e["annee"] and i != i_source]
        if not anterieures:
            non_resolus_copies.append((e, texte, match, "pas d'édition antérieure affichée pour ce graveur"))
            continue
        an_cible, i_cible = max(anterieures)
        fleches_copies.append({
            "i_source": i_source, "i_cible": i_cible,
            "graveur_source": e["graveur"], "graveur_cible": match,
            "an_source": e["annee"], "an_cible": an_cible,
        })

# Plusieurs flèches de copie peuvent partager un même point — le plus souvent en origine (un
# graveur souvent copié, comme un Giacomo Franco ou un Bernard Salomon, cité comme source par
# plusieurs éditions différentes), parfois en arrivée (une édition qui copie plusieurs
# graveurs à la fois). Sans correctif, ces flèches partiraient (ou arriveraient) toutes avec
# la même courbure et se chevaucheraient près du point commun. On regroupe donc les flèches
# par point partagé (origine ET arrivée) et on leur donne, comme les flèches de circulation de
# 01_carte_circulation.ipynb, une courbure différente (signe alterné, amplitude croissante).
def courbure_pour_rang(rang):
    amplitude = 0.16 + 0.1 * (rang // 2)
    signe = 1 if rang % 2 == 0 else -1
    return amplitude * signe

groupes_par_point = {}
for idx, f in enumerate(fleches_copies):
    groupes_par_point.setdefault(f["i_cible"], []).append(idx)
    groupes_par_point.setdefault(f["i_source"], []).append(idx)

for idx, f in enumerate(fleches_copies):
    rang_max = max(
        groupes_par_point[f["i_cible"]].index(idx),
        groupes_par_point[f["i_source"]].index(idx),
    )
    f["bulge"] = courbure_pour_rang(rang_max)

def courbe_copie(x1, y1, x2, y2, bulge):
    """Chemin SVG en courbe douce (quadratique) plutôt qu'un trait droit : une flèche de copie
    relie souvent deux lignes différentes, une courbe reste lisible en s'écartant des lignes
    intermédiaires au lieu de les traverser en diagonale sèche."""
    dx, dy = x2 - x1, y2 - y1
    dist = math.hypot(dx, dy) or 1
    nx, ny = -dy / dist, dx / dist  # perpendiculaire unitaire (toujours du même côté)
    cx = (x1 + x2) / 2 + nx * dist * bulge
    cy = (y1 + y2) / 2 + ny * dist * bulge
    return f'M {x1:.1f} {y1:.1f} Q {cx:.1f} {cy:.1f} {x2:.1f} {y2:.1f}'

# Tracée de l'édition copiée (plus ancienne) vers l'édition copieuse (plus récente), dans le
# même sens chronologique (gauche -> droite) que les segments de réemploi — contrairement aux
# flèches de copie de 01_carte_circulation.ipynb (où la pointe part de la plus récente vers la
# plus ancienne pour montrer "ce que copie cette édition") : ici, sur une frise où toutes les
# autres flèches lisent le temps de gauche à droite, ce sens inverse se lisait à l'envers.
for f in fleches_copies:
    x1, y1 = points[f["i_cible"]]["fx"], points[f["i_cible"]]["fy"]
    x2, y2 = points[f["i_source"]]["fx"], points[f["i_source"]]["fy"]
    x2r, y2r = retrait_vers(x1, y1, x2, y2, RAYON_POINT + 4)
    titre_infobulle = f'{f["graveur_source"]} ({f["an_source"]}) copie {f["graveur_cible"]} ({f["an_cible"]})'
    elements_svg.append(
        f'<path d="{courbe_copie(x1, y1, x2r, y2r, f["bulge"])}" class="fleche-copie" marker-end="url(#fleche-copie-tete)">'
        f'<title>{titre_infobulle}</title></path>'
    )

print(len(fleches_copies), "flèches de copie affichées")
print(len(non_resolus_copies), "mentions non résolues ou non affichables :")
for e, texte, cible, raison in non_resolus_copies[:15]:
    print(f"  ✗ {e['graveur']:20s} {e['annee']} — {texte!r} — {cible!r} ({raison})")
if len(non_resolus_copies) > 15:
    print(f"  … et {len(non_resolus_copies) - 15} de plus")

# --- Cercles des points, ajoutés en dernier pour rester au-dessus des segments et des
# flèches de copie ---
for i, pt in enumerate(points):
    elements_svg.append(f'<circle class="point-frise" data-i="{i}" cx="{pt["fx"]}" cy="{pt["fy"]}" r="{RAYON_POINT}"/>')

FRISE_SVG_CONTENU = "\n".join(elements_svg)
print(len(points), "points affichés,", HAUTEUR, "px de haut")

# --- Tableau détaillé : une ligne par édition, groupée par jeu de plaques (même ordre que la frise) ---
def lignes_tableau_plaque(p):
    lignes = []
    for j, e in enumerate(p["editions"]):
        lien_html = f'<a href="{e["lien"]}" target="_blank">voir</a>' if e["lien"] else ""
        graveur_cell = p["graveur"] if j == 0 else ""
        lignes.append(
            f'<tr><td>{graveur_cell}</td><td>{e["annee"]}</td><td>{e["ville"]}</td>'
            f'<td>{e["publisher"]}</td><td>{e["titre"]}</td>'
            f'<td>{LIBELLES_TECHNIQUE[e["technique"]]}</td><td>{lien_html}</td></tr>'
        )
    return lignes

LIGNES_TABLEAU = "\n".join(l for p in plaques for l in lignes_tableau_plaque(p))

In [ ]:
TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Réemploi des plaques gravées entre éditeurs</title>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6; --c-transfert: #c0392b; --c-copie: #1d4e74; --bande: #f2e9d8;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5; --c-transfert: #e0685a; --c-copie: #7fb3dd; --bande: #232019;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:none; margin:0; padding:16px 24px 32px; box-sizing:border-box; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 12px; }

  .legende { display:flex; gap:18px; flex-wrap:wrap; font-size:12px; margin:0 0 14px; }
  .legende .item { display:flex; align-items:center; gap:6px; }
  .legende .trait { display:inline-block; width:26px; height:0; border-top-width:2px; }
  .legende .reprise { border-top:2px solid var(--texte-att); }
  .legende .transfert { border-top:2px solid var(--c-transfert); }
  .legende .incertain { border-top:2px dashed var(--texte-att); }
  .legende .copie { border-top:2px dashed var(--c-copie); }

  /* Pleine largeur de la page, pas de barre de défilement interne : le SVG est responsive
     (width:100%, viewBox conservé) et s'étire pour occuper tout l'espace disponible du
     navigateur, au lieu d'être coincé dans une boîte à largeur fixe. */
  .cadre-frise { border:1px solid var(--trait); border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.15); }
  .frise { display:block; width:100%; height:auto; }
  .bande-zebra { fill:var(--bande); }
  .etiquette-plaque { font-size:11px; fill:var(--texte-fort); }
  .grille-frise { stroke:var(--trait); stroke-width:1; }
  .etiquette-annee-frise { font-size:10px; fill:var(--texte-att); }
  .segment-reprise { stroke:var(--texte-att); stroke-width:1.5; }
  .segment-transfert { stroke:var(--c-transfert); stroke-width:2.5; }
  .segment-incertain { stroke:var(--texte-att); stroke-width:1.5; stroke-dasharray:3,3; }
  /* Flèches de copie : bleu foncé, bien distinct du rouge de transfert et du bleu clair des
     liens/points survolés (--c-lien), assez marqué pour se voir sans dominer la frise. */
  .fleche-copie { fill:none; stroke:var(--c-copie); stroke-width:1.8; stroke-dasharray:2,4;
    opacity:.9; }
  .point-frise { fill:var(--surface); stroke:var(--contour-point); stroke-width:1.6; cursor:pointer;
    transition:r .15s; }
  .point-frise:hover, .point-frise.actif { r:9; fill:var(--c-lien); }

  .action-tableau { margin:14px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { width:100%; border-collapse:collapse; font-size:12px; margin:8px 0;
    display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-lien); }

  .infobulle { position:absolute; pointer-events:none; background:var(--surface);
    border:1px solid var(--contour-point); border-radius:5px; padding:6px 10px; font-size:12px;
    max-width:260px; opacity:0; transition:opacity .1s; box-shadow:0 2px 8px rgba(0,0,0,.3); z-index:2000; }
  .infobulle.epinglee { pointer-events:auto; }
  .infobulle a { color:var(--c-lien); }
  .infobulle .fermer-infobulle { position:absolute; top:2px; right:6px; cursor:pointer;
    color:var(--texte-att); font-size:13px; }
</style></head><body>
<div class="page">
  <h1>Réemploi des plaques gravées entre éditeurs</h1>
  <p class="souschapo">Une ligne = un jeu de plaques, un point = une édition qui l'utilise,
    positionnée par année. La plupart des lignes réunissent au moins deux éditions réemployées ;
    quelques-unes n'en montrent qu'une seule, ajoutée pour ancrer une flèche de copie vers ou
    depuis un autre jeu de plaques. Cliquer un point affiche le détail avec le lien "voir" (le
    survol seul n'affiche qu'un aperçu).</p>
  <div class="legende">
    <div class="item"><span class="trait reprise"></span>réimpression (même éditeur)</div>
    <div class="item"><span class="trait transfert"></span>transmission à un autre éditeur</div>
    <div class="item"><span class="trait incertain"></span>incertain (éditeur non identifié)</div>
    <div class="item"><span class="trait copie"></span>copie des illustrations d'un autre jeu de plaques</div>
  </div>
  <div class="cadre-frise">
    <svg class="frise" viewBox="0 0 __LARGEUR__ __HAUTEUR__">
      <defs>
        <marker id="fleche-transfert" viewBox="0 0 10 10" refX="8" refY="5"
          markerWidth="6" markerHeight="6" orient="auto-start-reverse">
          <path d="M0,0 L10,5 L0,10 z" fill="var(--c-transfert)"/>
        </marker>
        <marker id="fleche-copie-tete" viewBox="0 0 10 10" refX="8" refY="5"
          markerWidth="5" markerHeight="5" orient="auto-start-reverse">
          <path d="M0,0 L10,5 L0,10 z" fill="var(--c-copie)"/>
        </marker>
      </defs>
      __FRISE_SVG__
    </svg>
  </div>
  <div class="action-tableau">
    <button class="bascule" id="boutonTableau">Afficher le tableau détaillé</button>
    <table class="tableau-detaille" id="tableauDetaille">
      <thead><tr><th>Plaques</th><th>Année</th><th>Ville</th><th>Éditeur</th><th>Titre</th><th>Technique</th><th>Lien</th></tr></thead>
      <tbody>
        __LIGNES_TABLEAU__
      </tbody>
    </table>
  </div>
  <div class="infobulle" id="infobulle"></div>
</div>
<script>
  const points = __POINTS__;
  const libellesTechnique = __LIBELLES__;

  // Deux variantes du contenu, comme dans 02_nuage_editions_villes.ipynb : l'aperçu au
  // survol n'inclut PAS le lien (il ne serait pas cliquable, l'infobulle de survol ignorant
  // les clics tant qu'elle n'est pas épinglée) ; la version détaillée, affichée au clic une
  // fois épinglée, inclut le lien "voir".
  function contenuApercu(p) {
    return '<b>' + p.graveur + '</b><br>' +
      '<span>' + p.ville + ', ' + p.annee + ' · ' + libellesTechnique[p.technique] + '</span>' +
      '<br><i>' + p.editeur + '</i>' +
      (p.titre ? '<br>' + p.titre : '');
  }
  function contenuDetaille(p) {
    const lien = p.lien ? '<br><a href="' + p.lien + '" target="_blank">→ voir</a>' : '';
    return contenuApercu(p) + lien;
  }

  const infobulle = document.getElementById('infobulle');
  const page = document.querySelector('.page');
  let infobulleEpinglee = false;

  function positionnerInfobulle(ev) {
    const r = page.getBoundingClientRect();
    infobulle.style.left = (ev.clientX - r.left + 14) + 'px';
    infobulle.style.top = (ev.clientY - r.top + 14) + 'px';
  }
  function fermerInfobulle() {
    infobulleEpinglee = false;
    infobulle.classList.remove('epinglee');
    infobulle.style.opacity = 0;
    document.querySelectorAll('.point-frise.actif').forEach(c => c.classList.remove('actif'));
  }

  document.querySelectorAll('.point-frise').forEach(cercle => {
    const p = points[+cercle.dataset.i];
    cercle.addEventListener('mouseenter', () => {
      if (infobulleEpinglee) return;
      cercle.classList.add('actif');
      infobulle.innerHTML = contenuApercu(p);
      infobulle.style.opacity = 1;
    });
    cercle.addEventListener('mousemove', (ev) => {
      if (infobulleEpinglee) return;
      positionnerInfobulle(ev);
    });
    cercle.addEventListener('mouseleave', () => {
      if (infobulleEpinglee) return;
      cercle.classList.remove('actif');
      infobulle.style.opacity = 0;
    });
    cercle.addEventListener('click', (ev) => {
      ev.stopPropagation();
      positionnerInfobulle(ev);
      infobulle.innerHTML = contenuDetaille(p) + '<span class="fermer-infobulle" title="Fermer">×</span>';
      infobulle.style.opacity = 1;
      infobulle.classList.add('epinglee');
      infobulleEpinglee = true;
      document.querySelectorAll('.point-frise.actif').forEach(c => c.classList.remove('actif'));
      cercle.classList.add('actif');
      infobulle.querySelector('.fermer-infobulle').addEventListener('click', fermerInfobulle);
    });
  });

  document.addEventListener('click', (ev) => {
    if (infobulleEpinglee && !infobulle.contains(ev.target) && !ev.target.classList.contains('point-frise')) {
      fermerInfobulle();
    }
  });

  const boutonTableau = document.getElementById('boutonTableau');
  boutonTableau.addEventListener('click', () => {
    const tableau = document.getElementById('tableauDetaille');
    const visible = tableau.classList.toggle('visible');
    boutonTableau.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__LARGEUR__", str(LARGEUR))
    .replace("__HAUTEUR__", str(HAUTEUR))
    .replace("__FRISE_SVG__", FRISE_SVG_CONTENU)
    .replace("__LIGNES_TABLEAU__", LIGNES_TABLEAU)
    .replace("__POINTS__", json.dumps(points, ensure_ascii=False))
    .replace("__LIBELLES__", json.dumps(LIBELLES_TECHNIQUE, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Frise écrite dans", CHEMIN_SORTIE)